In [1]:
import napari
import os
import numpy as np
from skimage import io
from napari import layers
import pandas as pd
import glob
import dask.array as da
from dask_image import imread
import yaml
#from sahi import AutoDetectionModel, predict
import tqdm
import ngff_zarr as nz
import zarr
import json
from napari.utils.transforms import Affine

from multiview_stitcher import msi_utils
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion

In [2]:
viewer = napari.Viewer()

## Importing data

In [3]:
root = r'E:\Data\BiAPoL\Microwells\20250721\20250721_selectedPos_1_Focus Plane (order=2)_1'

viewer.layers.clear()
# list_of_files = [os.path.join(root, file) for file in os.listdir(root) if not 'converted' in file]
# _ = viewer.open(list_of_files, plugin='napari-autoscape')

In [4]:
folders = [os.path.join(root, folder) for folder in os.listdir(root) if os.path.isdir(os.path.join(root, folder))]
tif_files = glob.glob(os.path.join(root, '**', '*.tif'))
metadata = yaml.safe_load(open(os.path.join(root, folders[0], 'metadata.txt'), 'r'))['Summary']

In [5]:
positionlist = metadata['InitialPositionList']
positionlist = {position['Label']: position for position in positionlist}

## Stitching

In [6]:
tile_translations = []
tile_arrays = []
for label, position in tqdm.tqdm(positionlist.items()):
    z_scale = metadata['z-step_um'] if metadata['z-step_um'] != 0 else 1.0
    y_scale = metadata['PixelSize_um']
    x_scale = metadata['PixelSize_um']

    array = imread.imread(
        glob.glob(os.path.join(root, label, '*.tif'))[0]
    )

    #size_z = np.round(array.shape[0] * z_scale)
    size_y = np.round(array.shape[0] * y_scale)
    size_x = np.round(array.shape[1] * x_scale)

    tile_translations.append(
        {
            #"z": float(position["DeviceCoordinatesUm"]["ZStage:Z:32"][0]) - size_z // 2,
            "y": -float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][1]) - size_y // 2,
            "x": float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][0]) - size_x // 2
            }
    )
    tile_arrays.append(array)

100%|██████████| 323/323 [00:03<00:00, 88.68it/s]


In [7]:
y_min = min([translation['y'] for translation in tile_translations])
x_min = min([translation['x'] for translation in tile_translations])
y_max = max([translation['y'] + size_y for translation in tile_translations])
x_max = max([translation['x'] + size_x for translation in tile_translations])
print(f"y_min: {y_min}, y_max: {y_max}, x_min: {x_min}, x_max: {x_max}")

y_min: 3198.7138671875, y_max: 10344.7138671875, x_min: 41801.08203125, x_max: 48699.08203125


In [8]:
msims = []
for tile_array, tile_translation in zip(tile_arrays, tile_translations):
    sim = si_utils.get_sim_from_array(
        tile_array,
        dims=["c", "y", "x"],
        scale={"y": y_scale, "x": x_scale},
        translation=tile_translation,
        transform_key="stage_metadata",
    )
    msims.append(msi_utils.get_msim_from_sim(sim, scale_factors=[]))

fused_sim = fusion.fuse(
    [msi_utils.get_sim_from_msim(msim) for msim in msims],
    transform_key="stage_metadata",
    output_chunksize={'c': 1, 'y': 1528, 'x': 1528}
)[0]
#   .squeeze() \
#   .pad(x=1, y=1, constant_values=0)

In [9]:
ngff_image = nz.to_ngff_image(
    fused_sim,
    dims=["c", "y", "x"],
    translation={
        "y": y_min,
        "x": x_min,},
    scale={"y": y_scale, "x": x_scale},
    axes_units={"y": "micrometer", "x": "micrometer"},
    )

In [10]:
ngff_multiscales = nz.to_multiscales(
    ngff_image.data,
    scale_factors=[2, 4, 8, 16],
    method=nz.Methods.ITK_BIN_SHRINK)

In [11]:
ngff_multiscales

Multiscales(images=[NgffImage(data=dask.array<rechunk-merge, shape=(1, 23558, 21251), dtype=uint16, chunksize=(1, 128, 128), chunktype=numpy.ndarray>, dims=('z', 'y', 'x'), scale={'z': 1.0, 'y': 1.0, 'x': 1.0}, translation={'z': 0.0, 'y': 0.0, 'x': 0.0}, name='image', axes_units=None, axes_orientations=None, computed_callbacks=[<function _large_image_serialization.<locals>.<lambda> at 0x000001D7BD712E80>]), NgffImage(data=dask.array<empty_like, shape=(0, 11779, 10625), dtype=uint16, chunksize=(0, 128, 128), chunktype=numpy.ndarray>, dims=('z', 'y', 'x'), scale={'z': 2.0, 'y': 2.0, 'x': 2.0}, translation={'z': 0.5, 'y': 0.5, 'x': 0.5}, name='image', axes_units=None, axes_orientations=None, computed_callbacks=[]), NgffImage(data=dask.array<empty_like, shape=(0, 5889, 5312), dtype=uint16, chunksize=(0, 128, 128), chunktype=numpy.ndarray>, dims=('z', 'y', 'x'), scale={'z': 4.0, 'y': 4.0, 'x': 4.0}, translation={'z': 1.5, 'y': 1.5, 'x': 1.5}, name='image', axes_units=None, axes_orientations

In [ ]:
# Set memory target to 10000 MB
nz.to_ngff_zarr('test.ome.zarr', ngff_multiscales, use_tensorstore=False, version='0.5', chunks_per_shard=8)
viewer.open('test.ome.zarr', plugin='napari-ome-zarr')

## SCAPE visualization

In [24]:
root = r'E:\Data\BiAPoL\Microwells\20250721\SCAPE\20250721_Pos1-2'

In [33]:
import json
import pandas as pd

In [119]:
metadata_path = os.path.join(root, '20250721_Pos1-2_MMStack_Pos0_metadata.txt')
with open(metadata_path, 'r', encoding='latin-1') as f:
    json_data = json.load(f)

x = float(json_data['Summary']['Position_X'].replace(',', '').replace(' µm', '').replace('µm', ''))
y = -float(json_data['Summary']['Position_Y'].replace(',', '').replace(' µm', '').replace('µm', ''))
angle = float(json_data['Summary']['StageScanAnglePathA'])
pixelsize_xy = float(json_data['Summary']['PixelSize_um'])
z_step = float(json_data['Summary']['z-step_um'])
z_Step_corr = np.cos(angle * np.pi / 180) * z_step

slices_per_volume = int(json_data['Summary']['Slices'])

x, y, angle, pixelsize_xy, z_Step_corr

(44249.4, 5882.2, 49.0, 0.391, 0.32802951449525364)

In [200]:
affine_matrix = Affine(
    shear=np.array(
        [
            #0,
            0,
            np.tan(angle * np.pi / 180),
            #0, 
            0,
        ]),
    )

In [201]:
image_name = metadata_path.replace('_metadata.txt', '.ome.tif')
SCAPE_stack = imread.imread(image_name)[:slices_per_volume, :, :]

viewer = napari.Viewer(ndisplay=3, title=f'{affine_matrix.shear}')
layer = viewer.add_image(SCAPE_stack, name='SCAPE Stack', affine=affine_matrix, contrast_limits=[0, 3000], colormap='bop blue', axis_labels=['z', 'y', 'x'], blending='additive')
layer.bounding_box.visible = True

layer = viewer.add_image(SCAPE_stack, name='SCAPE Stack', contrast_limits=[0, 3000], colormap='bop orange', blending='additive', axis_labels=['z', 'y', 'x'])
layer.bounding_box.visible = True

viewer.camera.angles = (0, 0, 90)

In [114]:
image_name

'E:\\Data\\BiAPoL\\Microwells\\20250721\\SCAPE\\20250721_Pos1-2\\20250721_Pos1-2_MMStack_Pos1.ome.tif'

In [127]:
viewer.camera.angles = (0, 90, 0)
# viewer.camera.angles = (0, 0, 90)
#viewer.camera.angles = (90, 0, 0)

In [ ]:

image_stack

dask.array<_map_read_frame, shape=(502, 600, 600), dtype=uint16, chunksize=(1, 600, 600), chunktype=numpy.ndarray>

In [39]:
def find_problematic_byte(file_path, position):
    with open(file_path, 'rb') as file:  # Open in binary mode
        file.seek(position)  # Go to the problematic position
        # Read a small chunk around the problematic byte
        chunk = file.read(20)  # Read 20 bytes to see context
        print(f"Bytes around position {position}: {chunk}")

# Replace with your actual file path
file_path = os.path.join(root, '20250721_Pos1-2_MMStack_Pos1_metadata.txt')
find_problematic_byte(file_path, 2042)

Bytes around position 2042: b'\xb5m",\n  "Position_Y":'


## Object detection

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path='.runs/detect/train4/weights/yolo11n.pt',
    confidence_threshold=0.3,
    device="cuda:0",  # or 'cuda:0'
)

In [19]:
data = ngff_multiscales.images[0].data[1:-2, 1:-2]

# normalize to mean=126, std=64
data_normalized = (data - data.mean()) / data.std()  # standardize to mean=0, std=1
data = data_normalized * 64 + 126  # scale to mean=126, std=64
data = np.clip(data, 0, 255).astype(np.uint8)  # clip to valid range and convert to uint8
data = np.stack([data, data, data], axis=-1).compute().astype(np.uint8)

In [20]:
result = predict.get_prediction(data, detection_model=detection_model)

In [21]:
result.object_prediction_list

[]

In [8]:
rectangles = []
features = pd.DataFrame()
for layer in viewer.layers:
    if not isinstance(layer, layers.Image):
        continue

    io.imsave('image.png', layer.data[0][0].compute())
    prediction = model('image.png')
    for obj_class, box, conf in zip(prediction[0].boxes.cls, prediction[0].boxes.xywhn, prediction[0].boxes.conf):
        cls = int(obj_class)
        box = box.detach().cpu().numpy()

        z0 = layer.translate[0]
        y0 = layer.translate[1]
        x0 = layer.translate[2]
        img_size = layer.data.shape * layer.scale
        y0_box = box[1] * img_size[1]
        x0_box = box[0] * img_size[2]
        w = box[2] * img_size[2]
        h = box[3] * img_size[1]

        rectangle_corners = np.array([
            [z0, y0 + y0_box - h/2, x0 + x0_box - w/2],
            [z0, y0 + y0_box - h/2, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box - w/2]
        ])
        rectangle_corners[:, 1:] += box[:2] * layer.scale[1:]

        _features = pd.DataFrame({
            'y0': [y0_box],
            'x0': [x0_box],
            'w': [w],
            'h': [h],
            'cls': [cls],
            'conf': [conf.detach().cpu().numpy()],
            'area': [w * h],
        })
        features = pd.concat([features, _features], ignore_index=True)

    rectangles.append(rectangle_corners)


image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 1 cell, 35.0ms
Speed: 2.1ms preprocess, 35.0ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 5 cells, 20.7ms
Speed: 1.4ms preprocess, 20.7ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 7 cells, 21.2ms
Speed: 1.3ms preprocess, 21.2ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 3 cells, 1 spheroid, 21.0ms
Speed: 1.4ms preprocess, 21.0ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 3 cells, 20.6ms
Speed: 1.2ms preprocess, 20.6ms inference, 0.5ms postprocess per image at shape (1, 3, 512, 512)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 512x512 1 spheroid, 21.6ms
Speed: 1.3ms preprocess, 21.6ms i